# Proyecto Módulo 3

## Diplomado en Ciencia de Datos

## Aldo Alejandro Gallegos Ruiz

### Importación de Librerías

In [ ]:
import base64
import glob
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import seaborn as sns
import cufflinks as cf
from scipy.stats import zscore

from datetime import datetime
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
from io import BytesIO
from IPython.display import HTML
from mlxtend.frequent_patterns import association_rules, fpgrowth
from scipy.cluster.hierarchy import cut_tree, dendrogram, linkage
from scipy.stats import ks_2samp, zscore
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import KernelPCA, PCA, FastICA
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.impute import SimpleImputer
from sklearn.manifold import Isomap, MDS, TSNE
from sklearn.metrics import (calinski_harabasz_score, davies_bouldin_score, silhouette_score, silhouette_samples)
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances
from sklearn.mixture import GaussianMixture as GMM
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, PolynomialFeatures, StandardScaler
from varclushi import VarClusHi
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from scipy.cluster.hierarchy import cophenet
from scipy.spatial.distance import pdist
import plotly.express as px

cf.go_offline()
pd.set_option('display.max_columns', None)
pd.options.display.max_columns = None

# Carga de datos

In [ ]:
directorio = 'C:/Users/aldol/Programming/Diplo/bizilian_ecommerce'

patron_csv = '*.csv'

dataframes_csv = []

for archivo in glob.glob(os.path.join(directorio, patron_csv)):

    nombre_archivo = os.path.splitext(os.path.basename(archivo))[0]

    df_tempo = pd.read_csv(archivo)
    globals()[nombre_archivo] = df_tempo

### Importar librería de funciones

In [ ]:
from modulo3_proyecto_Libreria import proyecto_ecommerce
pec=proyecto_ecommerce()

## Información

In [ ]:
olist_customers_dataset.shape, olist_customers_dataset.columns

In [ ]:
olist_geolocation_dataset.shape, olist_geolocation_dataset.columns

In [ ]:
olist_order_items_dataset.shape, olist_order_items_dataset.columns

In [ ]:
olist_order_payments_dataset.shape, olist_order_payments_dataset.columns

In [ ]:
olist_order_reviews_dataset.shape, olist_order_reviews_dataset.columns

In [ ]:
olist_orders_dataset.shape, olist_orders_dataset.columns

In [ ]:
olist_products_dataset.shape, olist_products_dataset.columns

In [ ]:
olist_sellers_dataset.shape, olist_sellers_dataset.columns

# Tabla unificada

In [ ]:
merged_df = olist_order_payments_dataset.merge(olist_orders_dataset, on='order_id', how='inner') \
                  .merge(olist_order_reviews_dataset, on='order_id', how='inner') \
                  .merge(olist_order_items_dataset, on='order_id', how='inner') \
                  .merge(olist_customers_dataset,on='customer_id', how='inner') \
                  .merge(olist_products_dataset,on='product_id', how='inner') \
                    .merge(olist_sellers_dataset,on='seller_id', how='inner') \
                  .merge(olist_geolocation_dataset,left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix', how='inner')

In [ ]:
merged_df.isna().cumsum().tail(1)

# Limpieza de datos

## Pasar fechas a datetime

In [ ]:
merged_df=merged_df.copy()
cols=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date','shipping_limit_date']
for col in cols:
    merged_df[col]=pd.to_datetime(merged_df[col])
merged_df.dtypes

## Campos geolocation lat y lon

In [ ]:
merged_df=merged_df.drop(columns=['geolocation_lat', 'geolocation_lng']).drop_duplicates().reset_index(drop=True)

In [ ]:
merged_df.shape

## Imputacion de los valores ausentes

In [ ]:
merged_df.isna().cumsum().tail(1)

Debido a que todo producto tiene un nombre y descripción, resultaría conveniente imputar por la media

In [ ]:
var_long_prod=['product_name_lenght', 'product_description_lenght', 'product_photos_qty']
imputador_media = SimpleImputer(strategy='median')
merged_df[var_long_prod]=imputador_media.fit_transform(merged_df[var_long_prod])

### Imputacion por media para las variables de peso y tamaño

In [ ]:
var_tamano_peso=['product_weight_g',	'product_length_cm',	'product_height_cm',	'product_width_cm']
imputador_media = SimpleImputer(strategy='median')
merged_df[var_tamano_peso]=imputador_media.fit_transform(merged_df[var_tamano_peso])

### Imputacion por media para las variables de fecha

In [ ]:
merged_df.loc[merged_df['review_score']==1, 'review_comment_title'].value_counts()

In [ ]:
merged_df.loc[pd.isna(merged_df['order_approved_at'])]

In [ ]:
fechas_vacias=['order_approved_at',	'order_delivered_carrier_date',	'order_delivered_customer_date']

merged_df['order_approved_at']=merged_df['order_approved_at'].fillna(
    merged_df['order_purchase_timestamp']+(merged_df['order_approved_at']-merged_df['order_purchase_timestamp']).agg('mean'))

merged_df['order_delivered_carrier_date']=merged_df['order_delivered_carrier_date'].fillna(
    merged_df['order_purchase_timestamp']+(merged_df['order_delivered_carrier_date']-merged_df['order_purchase_timestamp']).agg('mean'))

merged_df['order_delivered_customer_date']=merged_df['order_delivered_customer_date'].fillna(
    merged_df['order_purchase_timestamp']+(merged_df['order_delivered_customer_date']-merged_df['order_purchase_timestamp']).agg('mean'))

## Resultado p-value de la prueba KS


Esto para demostrar que las variables imputadas no modificaron su distribución normal

In [ ]:
lista = []
for i in fechas_vacias+var_tamano_peso+var_long_prod:
    resultado_ks = ks_2samp(merged_df[i].dropna(), merged_df[i])
    lista.append([i,resultado_ks.pvalue])

In [ ]:
resultado_df = pd.DataFrame(lista)
resultado_df

# Precio mayor a cero

In [ ]:
merged_df.loc[merged_df['payment_value']==0, 'review_comment_message']

In [ ]:
merged_df=merged_df.loc[merged_df['payment_value']>0]

In [ ]:
merged_df.shape

# Variables continuas y discretas

In [ ]:
varc=['price',	'freight_value','product_name_lenght',	'product_description_lenght',	'product_photos_qty',	
      'product_weight_g',	'product_length_cm',	'product_height_cm',	'product_width_cm','seller_zip_code_prefix' , 
      'geolocation_zip_code_prefix', 'customer_zip_code_prefix']

vard=['payment_sequential','payment_installments','review_score','payment_type','order_status','customer_city',	'customer_state',
      'product_category_name','seller_city',	'seller_state','geolocation_city',	'geolocation_state']

### Información variables continuas

In [ ]:
varc_2=['price',	'freight_value',	'product_photos_qty',	'product_weight_g',	'product_length_cm','seller_zip_code_prefix' , 
      'geolocation_zip_code_prefix', 'customer_zip_code_prefix']

merged_df[varc_2].describe()

### Información variables discretas

In [ ]:
#pec.freq(merged_df, vard)

# Visualización gráfica

In [ ]:
pec.graficar_histogramas(merged_df,varc,2)

In [ ]:
pec.mapa_calor(merged_df)

In [ ]:
pec.graficar_bh(merged_df,['payment_type','order_status',	'customer_state','product_category_name',	'seller_state',	'geolocation_state'],2)

In [ ]:
pec.graficar_densidades(merged_df,varc,2)

# Variable objetivo

Con los datos con los que contamos buscaremos predecir el precio total que pagará el cliente por su compra

In [ ]:
np.std(merged_df['payment_value'])

In [ ]:
merged_df['payment_value'].describe()

In [ ]:
merged_df.shape

# Construcción de TAD

In [ ]:
merged_df=pec.construction_dat(merged_df)

---- Fin de la TAD ----

## Campo del Q

def asignar_cuatrimestre(mes):
        if mes in [1, 2, 3]:
            return 'Q1'
        elif mes in [4, 5, 6]:
            return 'Q2'
        elif mes in [7, 8, 9]:
            return 'Q3'
        elif mes in [10, 11, 12]:
            return 'Q4'

In [ ]:
merged_df['Quarter'] = merged_df.apply(lambda row: f"{pec.asignar_cuatrimestre(row['order_purchase_timestamp'].month)}-{row['order_purchase_timestamp'].year}", axis=1)

In [ ]:
merged_df.shape

# Pasar variables string a numéricas categóricas

In [ ]:
df_orig=merged_df.copy()

In [ ]:
granulares=['payment_type','order_status','customer_city','product_category_name',	'customer_state','seller_city',	'seller_state','geolocation_city',	'geolocation_state']

In [ ]:
label_encoder = LabelEncoder()

for g in granulares:
    merged_df[g] = label_encoder.fit_transform(merged_df[g])


## Variables predictoras

In [ ]:
tar='payment_value'
exclude=['order_id','customer_id','order_purchase_timestamp', 'review_creation_date', 'review_answer_timestamp',
          'order_approved_at', 'order_delivered_carrier_date',
         'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_id', 'order_item_id',
          'product_id', 'seller_id', 'shipping_limit_date', 'customer_unique_id', 'geolocation_lat', 'geolocation_lng',
           'review_comment_title',	'review_comment_message',	 tar,'Quarter']
predictors = [c for c in merged_df.columns if c not in exclude]

#  Remosión de valores infinitos

El dataframe no cuenta con valores infinitos 

In [ ]:
cols_with_infinity = merged_df[predictors].columns[~merged_df[predictors].apply(lambda x: np.isinf(x).any())].to_list()
print(f'Registros antes y después de remover infinitos: {merged_df[predictors].shape, merged_df[cols_with_infinity].shape}')

# Remosión de variables unarias

El dataframe no cuenta con variables unarias

In [ ]:
num_valores_unicos = merged_df[predictors].nunique()
columnas_a_mantener = num_valores_unicos[num_valores_unicos > 1].index
df_unaria = merged_df[columnas_a_mantener]
print(f'Las dimensiones de la tabla es:{merged_df[predictors].shape, df_unaria.shape}')

## Detección y remoción de valores extremos.

In [ ]:
pec.iqr(merged_df)

In [ ]:
pec.z_score(merged_df)

In [ ]:
pec.iso_forest(merged_df,predictors)

# Variables altamente correlacionadas.

No hay variables con correlación 1 en valor absoluto.

In [ ]:
df_corr=pd.DataFrame(merged_df[predictors+[tar]].corr()) 
df_corr['target_abs']=abs(df_corr['payment_value']) 
df_corr[['target_abs']].sort_values(by='target_abs',ascending=False)

In [ ]:
df_corr=pd.DataFrame(df_corr['target_abs'])
corr=df_corr.loc[df_corr['target_abs']>0.01].index.to_list()
merged_df_=merged_df[corr].drop_duplicates().reset_index(drop=True)

In [ ]:
predictors=[x for x in predictors if x not in ['product_category_name','product_photos_qty']]

In [ ]:
merged_df_.shape

## Identificación de clusters

def identificacion_clusters(df):
      vc = VarClusHi(df)
      vc.varclus() 
      conjuntos= vc.rsquare
      return(conjuntos)

In [ ]:
pec.identificacion_clusters(merged_df[predictors])

## Poder Predictivo con SelectKBest 

In [ ]:
mejores = pec.seleccionar_kbest(merged_df[predictors+[tar]],"payment_value",num_variables_deseadas=10)
mejores

# Modelación supervisada

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, r2_score

# Split entrenamiento y prueba

In [ ]:
seed_ = 200524

In [ ]:
X = merged_df_.drop(columns='payment_value').copy()
y = merged_df_[tar].copy()

Xt, Xv, yt, yv = train_test_split(X, y, test_size=.2, random_state=seed_)

In [ ]:
X.shape, Xv.shape

## Regresión Lineal

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
#Lineal
lr = LinearRegression(fit_intercept=True)
#lr.fit(Xt, yt)

In [ ]:
#y_p=lr.predict(Xt)
#print('Train:')
#print(f'MAE: {mean_absolute_error(yt, y_p)}')
#rint(f'MSE: {mean_squared_error(yt, y_p)}')
#print(f'R2: {r2_score(yt , y_p)}')

In [ ]:
#y_p=lr.predict(Xv)
#print('Validation:')
#print(f'MAE: {mean_absolute_error(yv, y_p)}')
#print(f'MSE: {mean_squared_error(yv, y_p)}')
#print(f'R2: {r2_score(yv , y_p)}')

# Árbol de decisión

In [ ]:
# Árboles:
#tree_reg = DecisionTreeRegressor(criterion='absolute_error', max_depth=5, min_samples_split=30, min_samples_leaf=12)

#tree_reg.fit(Xt, yt)

In [ ]:
#regressions = tree_reg.predict(Xt)
#print('Train:')
#print(f'MAE: {mean_absolute_error(yt, regressions)}')
#print(f'MSE: {mean_squared_error(yt, regressions)}')
#print(f'R2: {r2_score(yt, regressions)}')

In [ ]:
#regressions = tree_reg.predict(Xv)
#print('Val:')
#print(f'MAE: {mean_absolute_error(yv, regressions)}')
#print(f'MSE: {mean_squared_error(yv, regressions)}')
#print(f'R2: {r2_score(yv, regressions)}')

# Redes neuronales

In [ ]:
#tree_reg = MLPRegressor(hidden_layer_sizes=(5, 5, 5), batch_size=100, learning_rate='adaptive')
#tree_reg.fit(Xt, yt)

In [ ]:
#regressions = tree_reg.predict(Xt)
#print('Train:')
#print(f'MAE: {mean_absolute_error(yt, regressions)}')
#print(f'MSE: {mean_squared_error(yt, regressions)}')
#print(f'R2: {r2_score(yt, regressions)}')

In [ ]:
#regressions = tree_reg.predict(Xv)
#print('Val:')
#print(f'MAE: {mean_absolute_error(yv, regressions)}')
#print(f'MSE: {mean_squared_error(yv, regressions)}')
#print(f'R2: {r2_score(yv, regressions)}')

# Gradiente Boosting 

In [ ]:
#tree_reg = XGBRegressor(n_estimators=50, n_jobs=-1)
#tree_reg.fit(Xt, yt)

In [ ]:
#regressions = tree_reg.predict(Xt)
#print('Train:')
#print(f'MAE: {mean_absolute_error(yt, regressions)}')
#print(f'R2: {r2_score(yt, regressions)}')

In [ ]:
#regressions = tree_reg.predict(Xv)
#print('Val:')
#print(f'MAE: {mean_absolute_error(yv, regressions)}')
#print(f'R2: {r2_score(yv, regressions)}')

In [ ]:
from sklearn.linear_model import SGDClassifier, SGDRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVC, SVR
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Gradiente Descendiente Estocástico

In [ ]:
sgdr = make_pipeline(
    StandardScaler(),
    SGDRegressor(max_iter=100)
)
#sgdr.fit(Xt, yt)


#print(f'El MAE del entrenamiento es: {mean_absolute_error(yt, sgdr.predict(Xt))}')
#print(f'El MAE de la validación es: {mean_absolute_error(yv, sgdr.predict(Xv))}')

#print(f'El MSE del entrenamiento es: {mean_squared_error(yt, sgdr.predict(Xt))}')
#print(f'El MSE de la validación es: {mean_squared_error(yv, sgdr.predict(Xv))}')

#print(f'El R2 del entrenamiento es: {r2_score(yt, sgdr.predict(Xt))}')
#print(f'El R2 de la validación es: {r2_score(yv, sgdr.predict(Xv))}')

# SVM:

In [ ]:
svr = make_pipeline(
    StandardScaler(),
    SVR(kernel='rbf')
)
#svr.fit(Xt, yt)

#print(f'El MAE del entrenamiento es: {mean_absolute_error(yt, svr.predict(Xt))}')
#print(f'El MAE de la validación es: {mean_absolute_error(yv, svr.predict(Xv))}')

#print(f'El MSE del entrenamiento es: {mean_squared_error(yt, svr.predict(Xt))}')
#print(f'El MSE de la validación es: {mean_squared_error(yv, svr.predict(Xv))}')

#print(f'El R2 del entrenamiento es: {r2_score(yt, sgdr.predict(Xt))}')
#print(f'El R2 de la validación es: {r2_score(yv, sgdr.predict(Xv))}')

# Elastic net

In [ ]:
from sklearn.linear_model import Lasso, Lars, BayesianRidge, ElasticNet

In [ ]:
sc = StandardScaler()
#sc.fit(Xt)

models = {
    'net': ElasticNet(alpha=.5, l1_ratio=.5),
    'lasso': Lasso(alpha=1),
    'bayes': BayesianRidge()
}

for m in models:
    model = models.get(m)
    if m=='bayes':
        X_aux = pd.DataFrame(sc.transform(Xt), columns=Xt.columns)
        model.fit(X_aux, yt)
    else:
        model.fit(Xt, yt)

for m in models:
    print(m)
    model = models.get(m)
    if m=='bayes':
        datat = pd.DataFrame(sc.transform(Xt), columns=Xt.columns)
        datav = pd.DataFrame(sc.transform(Xv), columns=Xv.columns)
    else:
        datat = Xt.copy()
        datav = Xv.copy()
    
    predictions = model.predict(datat)
    print('Train:')
    print(f'MAE: {mean_absolute_error(yt, predictions)}')
    print(f'R2: {r2_score(yt, predictions)}')


    predictions = model.predict(datav)
    print('Validate:')
    print(f'MAE: {mean_absolute_error(yv, predictions)}')
    print(f'R2: {r2_score(yv, predictions)}')
    
    print('\n'*2)

# Monitoreo a través de Q's

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

In [ ]:
predictors_=merged_df_.columns.to_list()

In [ ]:
periods = {
    'Q4-2016': merged_df.loc[merged_df['Quarter']=='Q4-2016'],  

    'Q1-2017': merged_df.loc[merged_df['Quarter']=='Q1-2017'],
    'Q2-2017': merged_df.loc[merged_df['Quarter']=='Q2-2017'],
    'Q3-2017': merged_df.loc[merged_df['Quarter']=='Q3-2017'],
    'Q4-2017': merged_df.loc[merged_df['Quarter']=='Q4-2017'], 

    'Q1-2018': merged_df.loc[merged_df['Quarter']=='Q1-2018'],
    'Q2-2018': merged_df.loc[merged_df['Quarter']=='Q2-2018'],
    'Q3-2018': merged_df.loc[merged_df['Quarter']=='Q3-2018'],  
}

In [ ]:
list(periods.keys())

Variables predictoras que se usaron para el modelo

var_plot=['payment_sequential', 'payment_type', 'payment_installments','order_status', 'review_score', 'price', 'freight_value',
       'customer_zip_code_prefix', 'customer_city', 'customer_state','product_name_lenght', 'product_description_lenght', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm','seller_zip_code_prefix', 'seller_city', 'seller_state',
       'geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state','cantidad_vendedores_por_cliente', 'cantidad_productos_por_cliente',
       'cantidad_compras', 'cantidad_vendedores_por_compra','cantidad_productos_por_compra', 'rel_freight_payment',
       'median_freight_prod', 'max_freight_prod', 'suma_freight_prod','median_freight_cust', 'max_freight_cust', 'suma_freight_cust',
       'med_payment_sequential', 'max_payment_sequential','suma_payment_sequential', 'volumen_producto', 'med_volumen_prod',
       'max_volumen_prod', 'min_volumen_prod', 'med_peso_prod','max_peso_prod', 'min_peso_prod', 'pay_install_max', 'pay_install_prom',
       'pedidos_misma_ciudad', 'dif_purchase_delivered_carrier','dif_purchase_delivered_customer', 'dif_delivered_customer_estimated',
       'med_dif_purchase_delivered_carrier','min_dif_purchase_delivered_carrier','max_dif_purchase_delivered_carrier']

pec.plot_metrics(
    tree_reg, periods, var_plot, tar, 
    {'MAE': mean_absolute_error, 'MAPE': mean_absolute_percentage_error, 'MSE': mean_squared_error, 
     'R2': r2_score}
)

# -------------------------------------------------------------

# Modelación no supervisada

## Detección de Outliers

In [ ]:
df_outl=pec.outl_iso_forest(merged_df_,predictors)

# Estandarización

In [ ]:
var_no_dum=[x for x in df_outl.columns if x not in ['pedidos_misma_ciudad']]

In [ ]:
scaler =  StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df_outl[var_no_dum]),columns=df_outl[var_no_dum].columns).reset_index(drop=True)

In [ ]:
binary_df=pd.DataFrame(df_outl['pedidos_misma_ciudad']).reset_index(drop=True)

In [ ]:
df=pd.concat([df_std, binary_df],axis=1)

# Modelación no supervisada

# -------------------------
# Reducción de dimensiones

Utilizaremos el procedimiento de PCA para buscar reducir las deminesiones de nuestro DataFrame. Además de este método, se trató de utilizar KERNEL PCA y Análisis Facotrial, pero debido a la complejidad computacional, no se logró ejecutar los métodos.

Sin embargo, el método de PCA es bastante acertado para lograr determinar de mejor manera los componentes necesarios a los que reduciremos nuestro conjunto de datos. Nos basaremos en la varianza explicada para determinanr esta cifra.

# PCA

Veremos qué cantidad de componentes logran explicar la varianza del DataFrame. Buscamos que esta sea de al menos 0.7, pues buscamos aligerar más el DataFrame y que los métodos siguientes se puedan aplicar más rápidamente

In [ ]:
pca_20 = PCA(n_components=20)
pca_20.fit(df)

plt.grid()
plt.plot(np.cumsum(pca_20.explained_variance_ratio_))
plt.xlabel('Número de componentes')
plt.ylabel('Varianza Explicada')
plt.show()

In [ ]:
varianza_explicada_acumulada = np.cumsum(pca_20.explained_variance_ratio_)
print(f'La varianza explicada correspondiente a usar 20 componentes es: {varianza_explicada_acumulada[19]}')

In [ ]:
df_pca_20 = pd.DataFrame(pca_20.transform(df), columns=[f'PC{i+1}' for i in range(1,21)])

## Visualización en R2

In [ ]:
pca_2 = PCA(n_components=2)
pca_2.fit(df)

df_pca_2 = pd.DataFrame(pca_2.transform(df), columns=['PC1', 'PC2'])

In [ ]:
pca_20 = PCA(n_components=15)
pca_20.fit(df)

Elegimos 15 componentes para reducir la dimensionalidad, pues la cantidad de componenes mínima que cumple que la varianza explicada esté en por lo menos 0.75

In [ ]:
plt.figure(figsize=(5,5))
sp = sns.scatterplot(x='PC1', y='PC2', s=50, data=df_pca_2, color='orange').set_title("Visualización de grupos por PCA")
plt.show()

# ICA

In [ ]:
ica=FastICA(n_components=20,random_state=0)
df_ica=ica.fit_transform(df)
df_ica=pd.DataFrame(df_ica, columns=[f'IC{i+1}' for i in range(20)])

In [ ]:
corr_ica=df_ica.corr()
pec.mapa_calor(df_ica)
plt.show()

## Visualización en R2

In [ ]:
ica=FastICA(n_components=2,random_state=0)
df_ica_2=ica.fit_transform(df)
df_ica_2=pd.DataFrame(df_ica_2, columns=[f'IC{i+1}' for i in range(2)])

In [ ]:
df_ica_2

In [ ]:
plt.figure(figsize=(5,5))
sp = sns.scatterplot(x='IC1', y='IC2', s=50, data=df_ica_2, color='orange').set_title("Visualización de grupos por ICA")
plt.show()

# Kernel PCA

Para la ejecución de este proceso, mi computadora no logra almacenar la cantidad de infomación que se requería

In [ ]:
df.shape[0]

Sin embargo, tomaremos una muestra aleatoria del Dataframe

In [ ]:
df_sampled = df.iloc[np.random.choice(df.index, 1000, replace=False)]

In [ ]:
pca_2_kpca = KernelPCA(n_components=2, kernel='rbf')
pca_2_kpca.fit(df_sampled)

df_kpca_2 = pd.DataFrame(pca_2_kpca.transform(df), columns=['PC1', 'PC2'])

In [ ]:
plt.figure(figsize=(5,5))
sp = sns.scatterplot(x='PC1', y='PC2', s=50, data=df_kpca_2, color='orange').set_title("Visualización de grupos por Kernel PCA")
plt.show()

# t-SNE 

In [ ]:
tsne = TSNE(n_components=2, random_state=0)

df_tsne = pd.DataFrame(tsne.fit_transform(df))
df_tsne.columns=["componente_1","componente_2"]

In [ ]:
fig = px.scatter(df_tsne, x="componente_1", y="componente_2")
fig.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=0, perplexity = 50, n_iter = 5000)

df_tsne = pd.DataFrame(tsne.fit_transform(df))
df_tsne.columns=["componente_1","componente_2"]

In [ ]:
fig = px.scatter(df_tsne, x="componente_1", y="componente_2")
fig.show()

In [ ]:
tsne.kl_divergence_

In [ ]:
tsne.perplexity

In [ ]:
tsne.n_iter_

# MDS 

In [ ]:
mds = MDS(random_state=0)

In [ ]:
df_mds = pd.DataFrame(mds.fit_transform(df_sampled))

In [ ]:
df_mds

In [ ]:
#Matriz de distancias
mds.dissimilarity_matrix_

In [ ]:
#Métrica de distancias
mds.dissimilarity

In [ ]:
#Nuevos valores sobre el nuevo plano
mds.embedding_

**Las incrustaciones se crean en función del algoritmo de minimización de estrés**

In [ ]:
#Estadistico de bondad de ajuste
mds.stress_

In [ ]:
fig = plt.figure(2, (10,4))
ax = fig.add_subplot(122)
plt.scatter(df_mds[[0]].values, df_mds[[1]].values, s=65, c='r')
plt.title('Nueva Posición - Distancia Euclideana')
fig.subplots_adjust(wspace=.4, hspace=0.5)
plt.show()

In [ ]:
stress = mds.stress_
print(stress)

- Otro método para aplicar MDS es construir una matriz de distancia y aplicar MDS directamente a esta matriz

# MDS 2 

In [ ]:
dist_manhattan = manhattan_distances(df_sampled)

In [ ]:
dist_manhattan

In [ ]:
mds = MDS(dissimilarity='precomputed', random_state=0)

In [ ]:
#Conocer la nueva posición de los puntos
df_mds_1= mds.fit_transform(dist_manhattan)

In [ ]:
df_mds_1=pd.DataFrame(df_mds_1)

In [ ]:
stress = mds.stress_
stress

In [ ]:
fig = plt.figure(2, (10,4))
ax = fig.add_subplot(122)
plt.scatter(df_mds_1[[0]].values, df_mds_1[[1]].values, s=64, c='green')
plt.title('Nueva Posición - Distancia Manhattan')
fig.subplots_adjust(wspace=.4, hspace=0.5)
plt.show()

# Análisis Factorial

### PASO 1: Prueba de esfericidad de Bartlett

In [ ]:
chi2,p = calculate_bartlett_sphericity(df)
print("Esfericidad de Bartlett")
print("Valor de Chi : ",chi2)
print("P - value : ",p) 


Dado que el estadístico de la prueba p es inferior a 0,05, podemos decir que existe correlación entre las variables

### PASO 2: Prueba KMO

Se esperan valores mayores a 0.6 ya que representa que existe mayor correlación entre las variables allí dando paso a la aplicación de técnicas de reducción de dimensionalidad como el Análisis Factorial

In [ ]:
kmo_all,kmo_model = calculate_kmo(df)
print("KMO Test Statisitc",kmo_model)

# ------------------------
## Definicion de clusters

## Métodos visuales

Una vez teniendo un DataFrame más reducido,  nos apoyaremos de estos métodos visuales para determinar la cantidad óptima de clusters que debe de tener el modelo que vayamos a utilizar

### CODO - KMeans

In [ ]:
model = KMeans(init='k-means++', max_iter=50, n_init=10)
visualizer = KElbowVisualizer(model, k=(1,8)).fit(df_pca_20)
visualizer.show()

### Silueta - K-Means

In [ ]:
df_pca_20_sampled=df_pca_20.iloc[np.random.choice(df_pca_20.index, 1000, replace=False)]
df_pca_20_sampled=df_pca_20_sampled.reset_index(drop=True)

In [ ]:
silueta_kmeans = []
for k in list(range(2,20)):
    model = KMeans(n_clusters=k, init='k-means++', max_iter=50, n_init=10)
    model.fit(df_pca_20_sampled)
    labels = model.fit_predict(df_pca_20_sampled)
    score = silhouette_samples(df_pca_20_sampled, labels)
    score_avg = silhouette_score(df_pca_20_sampled, labels)
    silueta_kmeans.append(score_avg)

plt.plot(range(2,20), silueta_kmeans, 'bo-', label='k_means')

silueta_gmm = []
for k in list(range(2,20)):
    model = GMM(n_components=k, max_iter=100, n_init=10)
    model.fit(df_pca_20_sampled)
    labels = model.fit_predict(df_pca_20_sampled)
    score = silhouette_samples(df_pca_20_sampled, labels)
    score_avg = silhouette_score(df_pca_20_sampled, labels)
    silueta_gmm.append(score_avg)

plt.plot(range(2,20), silueta_gmm, 'go-', label='GMM')

plt.xlabel('Número de clusters')
plt.ylabel('Coeficiente de silueta')
plt.title('Coeficiente de silueta para GMM y KMeans')
plt.legend()
plt.show()

In [ ]:
for i in range(2,5):
    km = KMeans(n_clusters=i, init='k-means++', max_iter=50, n_init=10)
    visualizer = SilhouetteVisualizer(km, colors='yellowbrick')
    visualizer.fit(df_pca_20_sampled)
    print(f"N Clusters : {i}")
    print(f"Score Silueta : {round(visualizer.silhouette_score_,2)}")
    visualizer.show()

### CALINSKI

GMM

In [ ]:
calinski_gmm = []
for k in list(range(2, 21)):
    model = GMM(n_components=k, random_state=42)
    model.fit(df_pca_20)
    labels = model.fit_predict(df_pca_20)
    score = calinski_harabasz_score(df_pca_20,labels)
    calinski_gmm.append(score)

plt.xlabel('Número de clusters')
plt.ylabel('Coeficiente de Calinski Harabasz')
plt.title('Coeficiente de Calinski Harabasz para GMM')
plt.plot(range(2,21), calinski_gmm, 'go-', label='GMM')
plt.show()

K-Means

In [ ]:
calinski_kmeans = []
for k in list(range(2, 21)):
    km = KMeans(n_clusters=k, init='k-means++', max_iter=50, n_init=10)
    km.fit(df_pca_20)
    labels = km.fit_predict(df_pca_20)
    score = calinski_harabasz_score(df_pca_20,labels)
    calinski_kmeans.append(score)

plt.xlabel('Número de clusters')
plt.ylabel('Coeficiente de Calinski Harabasz')
plt.title('Coeficiente de Calinski Harabasz para K-Means')
plt.plot(range(2,21), calinski_kmeans, 'bo-', label="k-means")
plt.show()

# Clustering Jerárquico

## Dendogramas 

In [ ]:
mergings = linkage(df_pca_20_sampled,method = "single", metric="euclidean")
fig = ff.create_dendrogram(mergings)

In [ ]:
fig

In [ ]:
# Revisar combinaciones

In [ ]:
metodos = ['ward', 'average', 'complete',"single"]
distancias = ["euclidean",  'chebyshev', "cosine"]


metodos_list = [[metodos[i], distancias[j]] for i in range(len(metodos))
                for j in range(len(distancias))]
distancias_list = [[metodos, distancias] for metodos, distancias in metodos_list
                if metodos != 'ward' or distancias == 'euclidean']

In [ ]:
for metodo,distancia in distancias_list:
    mergings = linkage(df_pca_20_sampled, method = metodo, metric=distancia)
    fig = ff.create_dendrogram(mergings)
    title=f"Metodo : {metodo.upper()} - Dist : {distancia.upper()}"
    fig.update_layout(width=1000, height=500,title=title)
    fig.show()

##  Correlación Cofenética

In [ ]:
distancias_list

In [ ]:
for metodo,distancia in distancias_list:
    title=f"Metodo : {metodo.upper()} - Dist : {distancia.upper()}"
    print(title)
    corr=pec.corr_cofenetico(df_pca_20_sampled,metodo,distancia)
    print(corr)

## Visualizaciones 

In [ ]:
metodos = ['ward', 'average', 'complete']
distancias = ["euclidean",  'manhattan', "cosine"]
cl=[[2,3,4],[2,3,6],[2,4],[2,4]]


opciones = [[metodos[i], distancias[j],cl[i]] for i in range(len(metodos))
                for j in range(len(distancias))]
opciones = [[metodos, distancias,cl] for metodos, distancias,cl in opciones
                if metodos != 'ward' or distancias == 'euclidean']

In [ ]:
opciones

In [ ]:
for metodo,metrica,n in opciones:

    for i in n:
        cl=AgglomerativeClustering(n_clusters=i,linkage=metodo,metric=metrica)
        df_pca_20_sampled["cl"]=cl.fit_predict(df_pca_20_sampled)
        name=f"{metodo.upper()} Compontentes : {i} Métrica : {metrica}"
        fig=pec.plot_clusters_3d(df_pca_20_sampled,"cl",name)
        fig.show()

## Agrupación Final 

In [ ]:
data=pd.DataFrame(sc.inverse_transform(X.drop(columns=["cl"])),columns=X.drop(columns=["cl"]).columns)

In [ ]:
cl=AgglomerativeClustering(n_clusters=3,linkage="ward",affinity='euclidean')
data["cl"]=cl.fit_predict(X_pca)

In [ ]:
corr_cofenetico(X_pca,"ward","euclidean")

In [ ]:
data["cl"].value_counts(1)

In [ ]:
data.groupby(["cl"]).describe()

In [ ]:
data["pais"]=country_values

In [ ]:
data[data["cl"]==2][["pais"]]

In [ ]:
data[data["cl"]==0][["pais"]]

In [ ]:
data[data["cl"]==1][["pais"]]

# ISOMAP

In [ ]:
mds = MDS(metric=True, random_state=0)
df_mds = pd.DataFrame(mds.fit_transform(df_pca_20_sampled))

In [ ]:
stress = mds.stress_
print(stress)

In [ ]:
isomap = Isomap(n_components=2,n_neighbors=5)
df_isomap = pd.DataFrame(isomap.fit_transform(df_pca_20_sampled))
df_isomap.shape

In [ ]:
df_isomap.columns=['PC1','PC2']

In [ ]:
isomap.embedding_

In [ ]:
plt.figure(figsize=(5,5))
sp = sns.scatterplot(x='PC1', y='PC2', s=50, data=df_isomap, color='orange').set_title("Visualización de grupos por ISOMAP")
plt.show()

# ------------
# Modelos

## K MEANS

In [ ]:
k_means=KMeans(n_clusters=3, init='random', max_iter=50, n_init=10,random_state=607)
k_means.fit(df_pca_20)
df_pca_2['cl_kmeans']=k_means.labels_

In [ ]:
df_pca_2['cl_kmeans'].value_counts(1)

### Centroides

In [ ]:
centroids = k_means.cluster_centers_
centroids

In [ ]:
score_db = davies_bouldin_score(df_pca_20, df_pca_2['cl_kmeans'])
score_ch = calinski_harabasz_score(df_pca_20, df_pca_2['cl_kmeans'])
print(f"Score Davies Bouldin : {round(score_db,2)}")
print(f"Score Calinski Harabasz : {round(score_ch,2)}")

In [ ]:
fig = px.scatter(df_pca_2, x='PC1', y='PC2', color='cl_kmeans',
                 color_continuous_scale='Peach',title = 'K-Means Clustering ').update_layout(plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', width=800, height=800)
fig

## DBSCAN

Obtendremos la cantidad óptima de vecinos

In [ ]:
n_neighbors = range(2, 20)
distances = []
for k in n_neighbors:
    neigh = NearestNeighbors(n_neighbors=k)
    nbrs = neigh.fit(df_pca_20)
    distances.append(np.mean(nbrs.kneighbors(df_pca_20)[0][:, -1]))

plt.plot(n_neighbors, distances)
plt.xlabel('Número de vecinos')
plt.ylabel('Distancia k-ésima más cercana')
plt.title('Gráfico de distancia k-ésima más cercana')
plt.show()

Obtendremos el épsilon óptimo

In [ ]:
neigh = NearestNeighbors(n_neighbors=7)
nbrs = neigh.fit(df_pca_20)
distancias,_ = nbrs.kneighbors(df_pca_20)
distancias = np.sort(distancias, axis=0)
distancias = distancias[:,1]
aux=pd.DataFrame()
aux["distancias"]=sorted(list(distancias),reverse=False)
aux["index"]=range(len(distancias))
fig = px.line(aux, x="index", y="distancias", title="Valor óptimo de Epsilon")
fig.show()

In [ ]:
dbscan=DBSCAN()
dbscan.fit(df_pca_20)
dbscan_opt=DBSCAN(eps=0.5	,min_samples=7)
dbscan_opt.fit(df_pca_20)
df_pca_2['cl_dbscan'] = dbscan_opt.labels_

In [ ]:
score_db = davies_bouldin_score(df_pca_20, df_pca_2['cl_dbscan'])
score_ch = calinski_harabasz_score(df_pca_20, df_pca_2['cl_dbscan'])
print(f"Score Davies Bouldin : {round(score_db,2)}")
print(f"Score Calinski Harabasz : {round(score_ch,2)}")

In [ ]:
fig2 = px.scatter(df_pca_2, x='PC1', y='PC2', color_continuous_scale='Peach',title = 'DBSCAN Clustering',
            color='cl_dbscan').update_layout(plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', width=800, height=800)
fig2

## GMM

In [ ]:
gmm = GMM(n_components=3).fit(df_pca_20)
labels_gmm = gmm.predict(df_pca_20)
df_pca_2['cl_gmm'] = labels_gmm

In [ ]:
df_pca_2['cl_gmm'].value_counts(1)

In [ ]:
score_db = davies_bouldin_score(df_pca_20, df_pca_2['cl_gmm'])
score_ch = calinski_harabasz_score(df_pca_20, df_pca_2['cl_gmm'])
print(f"Score Davies Bouldin : {round(score_db,2)}")
print(f"Score Calinski Harabasz : {round(score_ch,2)}")

In [ ]:
fig = px.scatter(df_pca_2, x='PC1', y='PC2', color='cl_gmm',
                 color_continuous_scale='Peach',title = 'GMM Clustering ').update_layout(plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', width=800, height=800)
fig

# Modelo Ganador

Gracias a la visualización gráfica que logramos tener de ver cómo segmenta los datos cada uno de los modelos, nos quedarenos con el KMeans, ya que por mucho, es que mejor segmentación logró a comparación de los otros modelos. Vimos como DBSCAN tuvo una pésima segmentación, mientras que GMM redujo bastante el tercer cluster

In [ ]:
df_outl = df_outl.reset_index(drop=True)
df_pca_2 = df_pca_2.reset_index(drop=True)

In [ ]:
df_cluster = pd.concat([df_outl,df_pca_2['cl_kmeans']],axis=1)
df_cluster=df_cluster.rename(columns={'cl_kmeans': 'cluster'})

In [ ]:
df.shape, df_pca_2.shape

# -----------------------
## Análisis de clusters

Ya que tenemos el modelo ganador analizaremos las caracteristcas de nuestros clusters respecto a la información de los campos del DataFrame original, es decir, de la información linmpia sin escalar. 

Nos ayudaremos del campo is_host_superhost para hacer una segementación más precisa de los Airbnb's y conocer más sobre la caracteriticas de los lugares a partir de cómo se agrupan por clusters y de los host que son super o no superhost 

In [ ]:
df_cluster.price.describe()

In [ ]:
df_cluster['rank_price']=''

df_cluster.loc[(df_cluster.price<=39),'rank_price']='low'
df_cluster.loc[(df_cluster.price>39)&(df_cluster.price<=69),'rank_price']='medium_low'
df_cluster.loc[(df_cluster.price>69)&(df_cluster.price<=121),'rank_price']='medium_high'
df_cluster.loc[(df_cluster.price>121),'rank_price']='high'

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(df_cluster['cluster'].unique()), figsize=(10,2))
tema = sns.color_palette("Blues", len(df_cluster['cluster'].unique()))

for i, cluster in enumerate(df_cluster['cluster'].unique()):
    cluster_data = df_cluster[df_cluster['cluster'] == cluster]
    counts = cluster_data['rank_price'].value_counts()
    sns.barplot(x=counts, y=counts.index, ax=axes[i], palette=tema)

    axes[i].set_xlabel("Count")
    axes[i].set_title(f"Cluster {cluster}")
    axes[i].set_xlim([0, max(counts)*1.1])
    axes[i].set_ylim([-0.5, len(counts)])

fig.tight_layout()
plt.show()

## Análisis de resultados por cluster

In [ ]:
df_cluster[df_cluster['cluster']==2]['rank_price'].value_counts()[['high','low']].sum()

In [ ]:
rt_high_0 = (df_cluster[df_cluster['cluster']==0]['rank_price'].value_counts()[['medium_high','medium_low','low']].sum())/(df_cluster[df_cluster['cluster']==0]['rank_price'].value_counts()['high'])
rt_high_1 = (df_cluster[df_cluster['cluster']==1]['rank_price'].value_counts()[['medium_high','medium_low','low']].sum())/(df_cluster[df_cluster['cluster']==1]['rank_price'].value_counts()['high'])
rt_high_2 = (df_cluster[df_cluster['cluster']==2]['rank_price'].value_counts()[['medium_high','medium_low','low']].sum())/(df_cluster[df_cluster['cluster']==2]['rank_price'].value_counts()['high'])
rt_low_0 = (df_cluster[df_cluster['cluster']==0]['rank_price'].value_counts()[['medium_high','medium_low','high']].sum())/(df_cluster[df_cluster['cluster']==0]['rank_price'].value_counts()['low'])
rt_low_1 = (df_cluster[df_cluster['cluster']==1]['rank_price'].value_counts()[['medium_high','medium_low','high']].sum())/(df_cluster[df_cluster['cluster']==1]['rank_price'].value_counts()['low'])
rt_low_2 = (df_cluster[df_cluster['cluster']==2]['rank_price'].value_counts()[['medium_high','medium_low','high']].sum())/(df_cluster[df_cluster['cluster']==2]['rank_price'].value_counts()['low'])

In [ ]:
print(rt_high_0, rt_high_2, rt_high_2, rt_low_0, rt_low_1, rt_low_2)

Vemos que el cluster 0 tiene muchos superhost en comparación de los que no lo son, mientras que el cluster 0 y 2 tienen más host que no son super

Graficaremos ahora una serie de campos del DataFrame original que son comunes entre sí con respecto a qué tanto representan en cada cluster

In [ ]:
df_cluster

In [ ]:
location = ['customer_city',	'customer_state', 'seller_city',	'seller_state']
payment = ['payment_sequential',	'payment_type',	'payment_installments']
dimentions = ['product_name_lenght',	'product_description_lenght',	'product_weight_g',	'product_length_cm',	'product_height_cm',	'product_width_cm'	]
dif_days = ['dif_purchase_delivered_carrier',	'dif_purchase_delivered_customer',	'dif_delivered_customer_estimated'	]

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1)
df_cluster[df_cluster['cluster'] == 0][['customer_city']].value_counts()[:10].plot.barh(ax=axs[0], xlim=(0, 1000), figsize=(10,5), sharey=True, title='Cluster 0', color='orange')
df_cluster[df_cluster['cluster'] == 1][['customer_city']].value_counts()[:10].plot.barh(ax=axs[1], xlim=(0, 1000), figsize=(10,5), sharey=True, title='Cluster 1', color='orange')
df_cluster[df_cluster['cluster'] == 2][['customer_city']].value_counts()[:10].plot.barh(ax=axs[2], xlim=(0, 1000), figsize=(10,5), sharey=True, title='Cluster 2', color='orange')
plt.show()

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1)
df_cluster[df_cluster['cluster'] == 0][['customer_state']].value_counts()[:10].plot.barh(ax=axs[0], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 0', color='green')
df_cluster[df_cluster['cluster'] == 1][['customer_state']].value_counts()[:10].plot.barh(ax=axs[1], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 1', color='green')
df_cluster[df_cluster['cluster'] == 2][['customer_state']].value_counts()[:10].plot.barh(ax=axs[2], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 2', color='green')
plt.show()

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1)
df_cluster[df_cluster['cluster'] == 0][['seller_state']].value_counts()[:10].plot.barh(ax=axs[0], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 0', color='orange')
df_cluster[df_cluster['cluster'] == 1][['seller_state']].value_counts()[:10].plot.barh(ax=axs[1], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 1', color='orange')
df_cluster[df_cluster['cluster'] == 2][['seller_state']].value_counts()[:10].plot.barh(ax=axs[2], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 2', color='orange')
plt.show()

In [ ]:
df_cluster[df_cluster['cluster'] == 1][['customer_city']].value_counts()[:10]

In [ ]:
df_orig['customer_city'].value_counts()[:10]

In [ ]:
df_orig['customer_state'].value_counts()[:10]

Observamos los siguiente:

El cluster 0 es en donde hay host que menos tienen cuartos registrados en la ciudad. 

El cluster 2 es en donde hay host que menos tienen casas registradas en la ciudad.

El cluster 1 es en donde mlos host tienen más registros de habitaciones y casas, cosa que concuerda con el ratio que calculamos anteriormente

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1, figsize=(10,22))
df_cluster[df_cluster['cluster'] == 0][['payment_type']].value_counts().plot.barh(ax=axs[0], xlim=(0, 5000), figsize=(10,5), sharey=True, title='Cluster 0', color='orange')
df_cluster[df_cluster['cluster'] == 1][['payment_type']].value_counts().plot.barh(ax=axs[1], xlim=(0, 10000), figsize=(10,5), sharey=True, title='Cluster 1', color='orange')
df_cluster[df_cluster['cluster'] == 2][['payment_type']].value_counts().plot.barh(ax=axs[2], xlim=(0, 10000), figsize=(10,5), sharey=True, title='Cluster 2', color='orange')
plt.show()

In [ ]:
df_orig['payment_type'].value_counts()

En cuanto a disponibilidad, reviews y el conteo de amenidades, los tres clusters son bastante similes

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1, figsize=(10,22))
df_cluster[df_cluster['cluster'] == 0][dimentions].mean().plot.barh(ax=axs[0], xlim=(0, 700), figsize=(10,5), sharey=True, title='Cluster 0', color='orange')
df_cluster[df_cluster['cluster'] == 1][dimentions].mean().plot.barh(ax=axs[1], xlim=(0, 700), figsize=(10,5), sharey=True, title='Cluster 1', color='orange')
df_cluster[df_cluster['cluster'] == 2][dimentions].mean().plot.barh(ax=axs[2], xlim=(0, 700), figsize=(10,5), sharey=True, title='Cluster 2', color='orange')
plt.show()

Veamos la media de score de los reviews que dieron los huéspedes en distintos rubros

In [ ]:
df_cluster

In [ ]:
print(df_cluster[df_cluster['cluster'] == 0]['review_score'].mean())
print(df_cluster[df_cluster['cluster'] == 1]['review_score'].mean())
print(df_cluster[df_cluster['cluster'] == 2]['review_score'].mean())

In [ ]:
print(df_cluster[df_cluster['cluster'] == 0]['payment_installments'].mean())
print(df_cluster[df_cluster['cluster'] == 1]['payment_installments'].mean())
print(df_cluster[df_cluster['cluster'] == 2]['payment_installments'].mean())

In [ ]:
print(df_cluster[df_cluster['cluster'] == 0]['cantidad_compras'].mean())
print(df_cluster[df_cluster['cluster'] == 1]['cantidad_compras'].mean())
print(df_cluster[df_cluster['cluster'] == 2]['cantidad_compras'].mean())

Los huespedes en el cluter 1 dieron reviews con rating más bajos, el cluster 0 y 2 son muy similares

In [ ]:
print(df_cluster[df_cluster['cluster'] == 0]['volumen_producto'].mean())
print(df_cluster[df_cluster['cluster'] == 1]['volumen_producto'].mean())
print(df_cluster[df_cluster['cluster'] == 2]['volumen_producto'].mean())

In [ ]:
print(df_cluster[df_cluster['cluster'] == 0]['freight_value'].mean())
print(df_cluster[df_cluster['cluster'] == 1]['freight_value'].mean())
print(df_cluster[df_cluster['cluster'] == 2]['freight_value'].mean())

El cluster 1 es el que más reviews tiene

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1, figsize=(10,5))
df_cluster[df_cluster['cluster'] == 0]['cantidad_vendedores_por_cliente'].value_counts().plot.bar(ax=axs[0], ylim=(0, 5000), sharey=True, title='Cluster 0', color='purple')
df_cluster[df_cluster['cluster'] == 1]['cantidad_vendedores_por_cliente'].value_counts().plot.bar(ax=axs[1], ylim=(0, 10000), sharey=True, title='Cluster 1', color='purple')
df_cluster[df_cluster['cluster'] == 2]['cantidad_vendedores_por_cliente'].value_counts().plot.bar(ax=axs[2], ylim=(0, 7000), sharey=True, title='Cluster 2', color='purple')
plt.show()


En cuanto a amenidades, el cluster 1 es el que mpas cantidad tiene y en más variadad, seguido por el cluster 0 y el 2

Sin embargo, el cluster 1 es el que menos tipos de cuartos tiene, mientas que el cluster 2 y 0 son los que más, pero el cluster 1 tiene más cuartos del tipo 0 que el 2. El room_type 0 son los que solo son de casas completas, y el room_type 2 son solo de "Hotel rooms"

In [ ]:
fix, axs = plt.subplots(ncols=3,nrows=1, figsize=(10,5))
df_cluster[df_cluster['cluster'] == 0]['payment_sequential'].value_counts().plot.bar(ax=axs[0], ylim=(0, 5000), sharey=True, title='Cluster 0', color='blue')
df_cluster[df_cluster['cluster'] == 1]['payment_sequential'].value_counts().plot.bar(ax=axs[1], ylim=(0, 5000), sharey=True, title='Cluster 1', color='blue')
df_cluster[df_cluster['cluster'] == 2]['payment_sequential'].value_counts().plot.bar(ax=axs[2], ylim=(0, 5000), sharey=True, title='Cluster 2',color='blue')
plt.show()

Finalmente, el cluster 1 tiene más tipos de propiedades y en más cantidad, mientras que los otros dos son similares

# -----------------------
## Perfilamiento

### Cluster 0 - Variedad de habitaciones

Los que se encuentran aquí se hosperados en un airbnb que menos proporción de superhost hay.

Sin embargo hay más variadad de tipo de habitaciones que el resto aunque en muy poca cantidad, pues en estos hay muy pocasa habitaciones registradas.

El precio que pagaron los huéspedes por estos lugares es de un nivel medio. Sin embargo, en general, se otorgan calificaciones menores, y hay menos interacción de los huéspedes

### Cluster 1 - Superhost

Cluster en el cual más cantidad de superhost vamos a tener, también son hospedajes en casa o departamentos completos

Estos tienen más amendiades y más tipos de propiedades, tienen un selecto tipo de habitaciones.

Los host de este cluster tienen más registros de habitaciones y propiedades en la ciudad.

Podría considerarse que estos son los host más buscados, pero también más cotizados, pues también es donde hay más interacción de los huepedes con las opiniones que dan.

### Cluster 2 - No superhost, precios económicos

Los huéspedes pagaron precios menores en estos lugares a comparación de los demás.

Hay menor variadad de amenidades, tipos de propiedades y aunque hay un poco más de registros de habiraciones.

Este se consierarái como el cluster donde los huéspedes menos pagaron y son los menos cotizados en la plataforma,  no por ello los que peor reviews tienen.

# ------------
# Estabilidad

Suponiendo que ha pasado el tiempo y hemos obtenido nuevos registros clasificados con el modelo original, es importante revisar la estabilidad de los clusters. Para ello, debemos comparar diferentes elementos a lo largo del tiempo.

## Comparación de centroides y análisis estadístico

In [ ]:
df_t1 = pd.DataFrame(df[:12000])
df_t2 = pd.DataFrame(df[12000:24000])
df_t3 = pd.DataFrame(df[24000:36000])
df_t4 = pd.DataFrame(df[36000:])

In [ ]:
dataframes = [df_t1, df_t2, df_t3, df_t4]
kmeans_models = {}
dict_centroids={}
results = []

for i, df in enumerate(dataframes, start=1):
    kmeans = KMeans(n_clusters=3, random_state=0).fit(df)
    labels = kmeans.labels_
    
    score_db = davies_bouldin_score(df, labels)
    score_ch = calinski_harabasz_score(df, labels)
    
    kmeans_models[f'kmeans{i}'] = kmeans 
    dict_centroids[f'centroid{i}'] = kmeans.cluster_centers_
    
    results.append({
        'DataFrame': f'df_t{i}',
        'Davies-Bouldin Score': score_db,
        'Calinski-Harabasz Score': score_ch
    })

results_df = pd.DataFrame(results)
print(results_df)
print(dict_centroids)

In [ ]:
for j in ['Davies-Bouldin Score','Calinski-Harabasz Score']:
    for i in range(2,5):
        pct_change=round(((results[i-1][j] - results[i-2][j]) / results[i-1][j])*100,2)
        print(f"{pct_change}% de cambio en el {j} en el tiempo {i} y {i-1}".format(pct_change))

In [ ]:
for i in range(2,5):
    distancia_cambio = np.linalg.norm(dict_centroids[f'centroid{i-1}'] - dict_centroids[f'centroid{i}'], axis=1)
    print(f"Distancia de Cambio en Centroides en el tiempo {i} y {i-1}: {distancia_cambio}")

In [ ]:
cambio_promedio_cen=distancia_cambio.mean()
print(f"En promedio los centroides se han movido: {round(cambio_promedio_cen,2)}")

In [ ]:
# Cambios superiores al 5% son significativos
# Cambios superiores al 10% es importante considerar generar una nueva segmentación

In [ ]:
# Lo ideal es que los centroides no cambien

## Cambio en la composición

In [ ]:
df_t1["Cluster"] = kmeans_models['kmeans1'].labels_
df_t2["Cluster"] = kmeans_models['kmeans2'].labels_
df_t3["Cluster"] = kmeans_models['kmeans3'].labels_
df_t4["Cluster"] = kmeans_models['kmeans4'].labels_

In [ ]:
df_t1["Cluster"].value_counts(1)

In [ ]:
df_t2["Cluster"].value_counts(1)

In [ ]:
df_t3["Cluster"].value_counts(1)

In [ ]:
df_t4["Cluster"].value_counts(1)

In [ ]:
proporcion_t1=df_t1[["review_score","Cluster"]].groupby("Cluster").count()/df_t1.shape[0]
proporcion_t2=df_t2[["review_score","Cluster"]].groupby("Cluster").count()/df_t2.shape[0]
proporcion_t3=df_t3[["review_score","Cluster"]].groupby("Cluster").count()/df_t3.shape[0]
proporcion_t4=df_t4[["review_score","Cluster"]].groupby("Cluster").count()/df_t4.shape[0]

In [ ]:
proporcion_t1.columns=["T1"]
proporcion_t2.columns=["T2"]
proporcion_t3.columns=["T3"]
proporcion_t4.columns=["T4"]

In [ ]:
proporcion=proporcion_t1.merge(proporcion_t2,left_index=True,right_index=True) \
                        .merge(proporcion_t3,left_index=True,right_index=True) \
                        .merge(proporcion_t4,left_index=True,right_index=True)

In [ ]:
for i in range(2,5):
    proporcion[f"cambio_T{i-1}_T{i}"]=proporcion[f'T{i}']-proporcion[f'T{i-1}']

In [ ]:
#Cambio en el tamaño que representan los grupos, mas evidente en el grupo 0 y grupo 1
proporcion

## Análisis de perfiles de segmentos

In [ ]:
#Evaluamos si las características demográficas y de comportamiento de los segmentos han cambiado.

In [ ]:
feats=df_t1.columns

In [ ]:
feats=feats[:-1].to_list()

In [ ]:
data_t1=data[:100]
data_t2=data[100:]

data_t1["Cluster"]=kmeans_t1.labels_
data_t2["Cluster"]=kmeans_t2.labels_

In [ ]:
# Análisis demográfico de los segmentos
cluster_profiles = df_t1[feats+["Cluster"]].groupby('Cluster').mean()
print(cluster_profiles)

In [ ]:
# Análisis demográfico de los segmentos
cluster_profiles = df_t2[feats+["Cluster"]].groupby('Cluster').mean()
print(cluster_profiles)

In [ ]:
# Análisis demográfico de los segmentos
cluster_profiles = df_t3[feats+["Cluster"]].groupby('Cluster').mean()
print(cluster_profiles)

In [ ]:
# Análisis demográfico de los segmentos
cluster_profiles = df_t4[feats+["Cluster"]].groupby('Cluster').mean()
print(cluster_profiles)

In [ ]:
# Las caracteristicas han cambiado significativamente, ya no son los mismos grupos

# -------------------------
# Market Basket Analysis

In [ ]:
market_df=merged_df[["customer_id","order_purchase_timestamp",'product_id']]
market_df["transaccion"] = market_df["customer_id"].astype(str)+"_"+market_df["order_purchase_timestamp"].astype(str)

market_df.head()

In [ ]:
market_df["product_id"].value_counts(1)

In [ ]:
prods=list(market_df["product_id"].value_counts().index)[:10]

In [ ]:
olist_products_dataset.loc[olist_products_dataset['product_id'].isin(prods)]

In [ ]:
market_df["aux"]=1

In [ ]:
basket_input=pd.pivot_table(market_df,index=["transaccion"],columns=["product_id"],values=["aux"],aggfunc=["sum"]).fillna(0)

In [ ]:
basket_input

In [ ]:
for i in basket_input:
    basket_input[i]=basket_input[i].map(lambda x:1 if x>0 else 0)

In [ ]:
basket_input.sum().sum()

In [ ]:
#for i in basket_input:
    #display(basket_input[i].value_counts())

In [ ]:
basket_input.columns=[x[2] for x in basket_input.columns]

In [ ]:
basket_input

In [ ]:
frequent_itemsets = fpgrowth(basket_input, min_support=0.0001, use_colnames=True)

In [ ]:
frequent_itemsets

In [ ]:
#Genera un DataFrame de reglas de asociación incluyendo el
#métricas 'puntuación', 'confianza' y 'elevación'
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules

#significa que estás eligiendo utilizar el lift como el criterio principal para evaluar la fuerza
#de las reglas de asociación generadas a partir de los conjuntos de ítems frecuentes.

In [ ]:
rules['lift'] = (rules['support'] / rules['antecedent support']) / rules['consequent support']

In [ ]:
rules

# Conclusiones

Se consiguió una buena segmentación del conjunto de datos, gracias a la limpeza del mismo. Con el AEX logrmas entneder mejor los cmapos que componen el DataFrame y saber con qué información estabamos trabajando. La dificultad de procesamiento no fue un impedimento, ya que el modelo con el que trabajamos funcionó bien, así mismo los métodos para determinar la cantidad de clusters fueron acertados en está elección.  

## t-SNE

## DBSCAN

## GMM

# Perfilamiento